In [0]:
import pyspark.sql.functions as F

In [0]:
schema = 'olist_ecommerce.retention.'
url = 'abfss://gold@adlsolistchurn2026.dfs.core.windows.net/'

In [0]:
olist_customer = spark.read.format('delta').load(f'{url}/olist_customers')
olist_customer.show(5)

In [0]:
olist_orders = spark.read.format('delta').load(url+'olist_order_records')
olist_orders.show(5)

In [0]:
temp_orders = olist_orders.alias('oo').join(olist_customer.alias('oc'), on='customer_id', how='inner')
ref_date = temp_orders.select(F.max('order_purchase_timestamp')).collect()[0][0]

In [0]:
customer_rfm = temp_orders.groupBy("customer_unique_id") \
            .agg(F.min('order_purchase_timestamp').alias('first_purchase'),
                 (F.date_diff(F.max('order_purchase_timestamp'),F.min('order_purchase_timestamp')) + 1)
                 .alias('tenure_days'),
                F.date_diff(F.lit(ref_date), F.max('order_purchase_timestamp')).alias('recency_days'),
                F.count("order_purchase_timestamp").alias('frequency_days'),
                F.sum('total_price').alias('monetary_value'))
customer_rfm.show(7)

In [0]:
rec_quant = customer_rfm.approxQuantile("recency_days", [0.2,0.4,0.6,0.8], 0.05)
customer_rfm = customer_rfm.withColumn('r_score', 
                                     F.when(F.col('recency_days') <= rec_quant[0], 5)
                                      .when(F.col('recency_days') <= rec_quant[1], 4)
                                      .when(F.col('recency_days') <= rec_quant[2], 3)
                                      .when(F.col('recency_days') <= rec_quant[3], 2)
                                      .otherwise(1))
freq_quant = customer_rfm.approxQuantile("frequency_days", [0.2,0.4,0.6,0.8], 0.05)
customer_rfm = customer_rfm.withColumn('f_score', 
                                     F.when(F.col('frequency_days') <= freq_quant[0], 1)
                                      .when(F.col('frequency_days') <= freq_quant[1], 2)
                                      .when(F.col('frequency_days') <= freq_quant[2], 3)
                                      .when(F.col('frequency_days') <= freq_quant[3], 4)
                                      .otherwise(5))
monetary_quant = customer_rfm.approxQuantile("monetary_value", [0.2,0.4,0.6,0.8], 0.05)
customer_rfm = customer_rfm.withColumn('m_score', 
                                     F.when(F.col('monetary_value') <= monetary_quant[0], 1)
                                      .when(F.col('monetary_value') <= monetary_quant[1], 2)
                                      .when(F.col('monetary_value') <= monetary_quant[2], 3)
                                      .when(F.col('monetary_value') <= monetary_quant[3], 4)
                                      .otherwise(5))
customer_rfm = customer_rfm.withColumn('rfm_score', F.col("r_score")+F.col("f_score")+F.col("m_score"))
customer_rfm = customer_rfm.withColumn('customer_value',
                                       F.when(F.col('rfm_score') >= 12, 'High Value')
                                      .when(F.col('rfm_score') >= 8, 'Medium Value')
                                      .when(F.col('rfm_score') >= 4, 'Low Value')
                                      .otherwise('Lost'))
customer_rfm.show(6)
customer_rfm.write.format('delta').mode('overwrite').saveAsTable(schema+'customer_rfm')